# Preprocesamiento del código para modelos de similitud

Este notebook toma los pares de programas generados en la etapa anterior y los transforma en representaciones más útiles para entrenar o evaluar modelos de detección de similitud.

La entrada principal son los archivos raw creados en el notebook 01: cada fila contiene dos fragmentos de código y una etiqueta que indica si el par es similar o no. Aquí no se cambia la etiqueta ni se crean nuevos pares; lo importante es preparar mejores versiones del texto y añadir variables que ayuden a comparar los programas.

La idea central es que el código fuente, tal como viene de los archivos originales, puede contener ruido: saltos de línea inconsistentes, comentarios, espacios sobrantes, nombres de variables distintos o literales específicos. Este notebook reduce parte de ese ruido sin destruir la estructura importante del programa.


## 1. Importación de librerías y configuración de visualización

Se importan las herramientas necesarias para leer archivos, manipular tablas, aplicar expresiones regulares y contar tokens.

Por qué importa este paso:

- pathlib permite manejar rutas de forma clara y portable.
- pandas facilita trabajar con los pares de código como tablas.
- re permite definir reglas de limpieza y tokenización.
- Counter permite medir solapamiento entre secuencias de tokens respetando repeticiones.

También se ajusta display.max_colwidth para que pandas muestre fragmentos más largos de código o tokens al inspeccionar ejemplos.


In [1]:
from pathlib import Path
import pandas as pd
import re
from collections import Counter

pd.set_option('display.max_colwidth', 120)

## 2. Definición y verificación de rutas

Se establecen las rutas hacia la carpeta de datos procesados y hacia los archivos raw de entrenamiento y validación.

Este paso es importante porque el notebook 02 depende directamente de las salidas del notebook 01. Antes de intentar cargar datos, se verifica que la carpeta y los CSV existan. Así se detectan errores de flujo de trabajo temprano, por ejemplo si todavía no se ejecutó la etapa anterior o si los archivos fueron movidos.


In [3]:
DATASET_PATH = Path('data')
PROCESSED_PATH = DATASET_PATH / 'processed'

TRAIN_RAW_PATH = PROCESSED_PATH / '01_train_model_raw.csv'
VAL_RAW_PATH = PROCESSED_PATH / '01_val_model_raw.csv'

print('PROCESSED_PATH existe:', PROCESSED_PATH.exists())
print('Train raw existe:', TRAIN_RAW_PATH.exists())
print('Val raw existe:', VAL_RAW_PATH.exists())


PROCESSED_PATH existe: True
Train raw existe: True
Val raw existe: True


### Interpretación del resultado

Las tres comprobaciones aparecen como True, lo que significa que la carpeta de datos procesados existe y que los archivos raw de entrenamiento y validación están disponibles.

Esto es relevante porque confirma que la etapa anterior del proyecto terminó correctamente. Si alguno de estos valores fuera False, el preprocesamiento no tendría una entrada confiable y habría que volver al notebook 01 antes de continuar.


## 3. Carga de datasets raw

Se leen los archivos 01_train_model_raw.csv y 01_val_model_raw.csv. Estos contienen pares de programas con código original y etiquetas.

La vista previa inicial permite confirmar tres cosas: que los archivos se cargaron, que el tamaño coincide con lo esperado y que las columnas tienen el contenido necesario para preprocesar.


In [4]:
train_raw = pd.read_csv(TRAIN_RAW_PATH)
val_raw = pd.read_csv(VAL_RAW_PATH)

print('Train:', train_raw.shape)
print('Validation:', val_raw.shape)

display(train_raw.head())


Train: (176, 10)
Validation: (44, 10)


,pair_id,language,scenario,file_1,file_2,path_1,path_2,code_1_raw,code_2_raw,label
0,train_pair_000127,java,train,071.java,021.java,data\train\java\071.java,data\train\java\021.java,\n\nimport java.io.*;\nimport java.net.*;\nimport java.util.*;\n\nimport javax.swing.text.html.*;\nimport javax.swin...,\n\n\n\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport misc.BASE64Encoder;\nimport javax.swing.*...,0
1,train_pair_000028,java,train,051.java,258.java,data\train\java\051.java,data\train\java\258.java,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,\n\nimport java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\n \npublic clas...,1
2,train_pair_000134,java,train,051.java,257.java,data\train\java\051.java,data\train\java\257.java,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,\n\nimport java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\n \npublic clas...,1
3,train_pair_000037,c,train,014.c,032.c,data\train\c\014.c,data\train\c\032.c,#include <stdio.h>\n#include <stdlib.h>\n#include <sys/time.h>\n#define MINCHAR 65\n#define MAXCHAR 122\n\n\n\nint ...,"\n\n#include <stdio.h>\n#include <stdlib.h>\n#include <strings.h>\n#include <ctype.h>\n\nint (){\n\tsystem(""wget -p ...",0
4,train_pair_000015,java,train,160.java,009.java,data\train\java\160.java,data\train\java\009.java,import java.io.*;\n\n\npublic class WatchDog\n{\npublic static void main (String[] args)\n{ String isdiff = ne...,\n\nimport java.util.*;\nimport java.text.*;\nimport java.io.*;\nimport java.*;\nimport java.net.*;\n\npublic class ...,0


### Interpretación del resultado

El conjunto de entrenamiento tiene 176 pares y el de validación tiene 44 pares. En conjunto suman 220 ejemplos, divididos aproximadamente en 80% para entrenamiento y 20% para validación.

Esta proporción es importante porque permite que el modelo aprenda con la mayor parte de los datos, pero reserva una parte independiente para revisar si lo aprendido generaliza. La vista previa también confirma que cada fila contiene dos archivos, sus rutas, el código raw de ambos y su etiqueta.


## 4. Validación de columnas obligatorias

Antes de transformar datos, se comprueba que existan las columnas mínimas requeridas: identificador del par, lenguaje, archivos, código raw y etiqueta.

Este paso tiene mucha importancia porque evita errores silenciosos. Si una columna falta, cualquier transformación posterior podría producir resultados incorrectos o fallar mucho más adelante, cuando sería más difícil encontrar la causa.


In [5]:
required_columns = [
    'pair_id',
    'language',
    'file_1',
    'file_2',
    'code_1_raw',
    'code_2_raw',
    'label'
]

for df_name, df in [('train_raw', train_raw), ('val_raw', val_raw)]:
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f'{df_name} no tiene estas columnas: {missing}')

print('Columnas validadas correctamente.')


Columnas validadas correctamente.


### Interpretación del resultado

El mensaje indica que todas las columnas necesarias están presentes. Esto significa que los datos tienen la estructura mínima para aplicar limpieza, tokenización, normalización y cálculo de métricas.

La relevancia de este control es que evita construir resultados sobre una tabla incompleta. Si faltara, por ejemplo, code_1_raw o label, el notebook no podría generar representaciones ni evaluar similitud de forma correcta.


## 5. Revisión de distribución de etiquetas y lenguajes

Se inspecciona la cantidad de ejemplos positivos y negativos, además de la distribución por lenguaje.

Esto importa porque el desempeño de un modelo depende de la composición del dataset. Un conjunto muy desbalanceado podría hacer que el modelo aprenda atajos, por ejemplo predecir siempre la clase mayoritaria. Revisar los lenguajes también ayuda a confirmar que entrenamiento y validación tienen una estructura comparable.


In [6]:
print('Distribución train:')
display(train_raw['label'].value_counts())

print('\nDistribución validation:')
display(val_raw['label'].value_counts())

print('\nLenguajes train:')
display(train_raw['language'].value_counts())

print('\nLenguajes validation:')
display(val_raw['language'].value_counts())


Distribución train:


label
0    88
1    88
Name: count, dtype: int64


Distribución validation:


label
1    22
0    22
Name: count, dtype: int64


Lenguajes train:


language
java    109
c        67
Name: count, dtype: int64


Lenguajes validation:


language
java    26
c       18
Name: count, dtype: int64

### Interpretación del resultado

La distribución de etiquetas está perfectamente balanceada: en entrenamiento hay 88 pares negativos y 88 positivos; en validación hay 22 negativos y 22 positivos.

Esto es muy relevante para modelado porque evita que el modelo favorezca una clase solo porque aparece más veces. También permite interpretar métricas futuras con más confianza.

La distribución por lenguaje muestra más ejemplos Java que C: 109 Java y 67 C en entrenamiento, 26 Java y 18 C en validación. Esto indica que el dataset no está balanceado por lenguaje, aunque ambos lenguajes sí están representados en las dos particiones. Por eso conviene revisar resultados por lenguaje en etapas posteriores.


## 6. Medición inicial de tamaño del código raw

La función add_raw_length_features agrega métricas simples del código original: número de caracteres y número de líneas para cada programa del par.

Estas métricas no modifican el código, pero sirven para entender el dataset. Por ejemplo, pares con tamaños extremadamente diferentes pueden ser más fáciles de clasificar como no similares, mientras que tamaños parecidos pueden requerir análisis más fino. También ayudan a detectar archivos vacíos o anormalmente grandes.


In [7]:
def add_raw_length_features(df):
    df = df.copy()
    df['code_1_raw_chars'] = df['code_1_raw'].fillna('').apply(len)
    df['code_2_raw_chars'] = df['code_2_raw'].fillna('').apply(len)
    df['code_1_raw_lines'] = df['code_1_raw'].fillna('').apply(lambda x: len(x.splitlines()))
    df['code_2_raw_lines'] = df['code_2_raw'].fillna('').apply(lambda x: len(x.splitlines()))
    return df

train_raw = add_raw_length_features(train_raw)
val_raw = add_raw_length_features(val_raw)

display(train_raw[['code_1_raw_chars', 'code_2_raw_chars', 'code_1_raw_lines', 'code_2_raw_lines']].describe())


,code_1_raw_chars,code_2_raw_chars,code_1_raw_lines,code_2_raw_lines
count,176.000000,176.000000,176.000000,176.000000
mean,2849.863636,2837.948864,116.977273,120.085227
std,2716.120955,2966.509713,93.440031,109.833841
min,212.000000,212.000000,12.000000,16.000000
25%,1216.750000,1222.000000,56.750000,58.000000
50%,2288.000000,2065.500000,96.000000,83.000000
75%,3528.750000,3311.000000,136.000000,137.000000
max,25470.000000,26088.000000,740.000000,776.000000


### Interpretación del resultado

Las estadísticas muestran que los programas tienen tamaños muy variados. En entrenamiento, el promedio ronda los 2,800 caracteres por código, pero existen archivos mucho más pequeños, desde 212 caracteres, y otros muy grandes, arriba de 25,000 caracteres.

Esto importa porque la longitud puede afectar tanto a modelos clásicos como a transformers. Códigos largos pueden aportar más señales, pero también más ruido y mayor costo computacional. Además, diferencias grandes de longitud entre dos programas pueden ser una pista de baja similitud, aunque no deben usarse como única evidencia.


## 7. Limpieza ligera del código

La función light_clean_code normaliza saltos de línea, elimina espacios al final de cada línea, reduce bloques excesivos de líneas vacías y recorta espacios al inicio y final.

La limpieza es intencionalmente ligera: no cambia nombres, instrucciones ni símbolos del lenguaje. Su objetivo es reducir diferencias superficiales de formato que no deberían dominar la comparación entre programas.

Por qué importa:

- Dos soluciones equivalentes pueden diferir solo por espacios o saltos de línea.
- Los modelos y métricas de texto pueden ser sensibles al formato.
- Mantener la estructura del código permite conservar información útil para etapas posteriores.


In [8]:
def light_clean_code(code):
    """
    Limpieza ligera del código.
    Conserva la estructura y los tokens importantes.
    """
    if not isinstance(code, str):
        return ''

    # Normalizar saltos de línea
    code = code.replace('\r\n', '\n').replace('\r', '\n')

    # Quitar espacios al final de cada línea
    code = '\n'.join(line.rstrip() for line in code.split('\n'))

    # Reducir líneas vacías excesivas
    code = re.sub(r'\n{3,}', '\n\n', code)

    # Quitar espacios extremos
    return code.strip()


## 8. Eliminación aproximada de comentarios

La función remove_c_java_comments elimina comentarios de bloque y comentarios de una línea usando expresiones regulares.

Los comentarios pueden contener explicaciones, nombres de estudiantes o texto irrelevante para la lógica del programa. Quitarlos ayuda a que la comparación se enfoque más en la estructura y los tokens ejecutables.

Es una aproximación, no un parser completo de C o Java. Aun así, para un preprocesamiento inicial es útil porque reduce ruido de manera simple y rápida.


In [9]:
def remove_c_java_comments(code):
    """
    Elimina comentarios // y /* */ sin intentar parsear el lenguaje completo.
    Es una aproximación razonable para preprocesamiento inicial.
    """
    if not isinstance(code, str):
        return ''

    # Eliminar comentarios multilínea /* ... */
    code = re.sub(r'/\*.*?\*/', ' ', code, flags=re.DOTALL)

    # Eliminar comentarios de una línea //...
    code = re.sub(r'//.*', ' ', code)

    return light_clean_code(code)


## 9. Definición de palabras clave y tokenización léxica

Se define un conjunto de palabras clave de C, C++ y Java, además de un patrón regular para dividir el código en tokens.

Un token es una unidad básica del código: puede ser una palabra clave, identificador, número, cadena, operador o signo de puntuación. Tokenizar permite comparar programas por sus elementos estructurales en lugar de tratarlos como texto libre.

Este paso importa porque muchos modelos y métricas funcionan mejor cuando el código se representa como una secuencia de unidades significativas.


In [10]:
JAVA_C_KEYWORDS = {
    # C / C++ / Java common
    'auto', 'break', 'case', 'char', 'const', 'continue', 'default', 'do',
    'double', 'else', 'enum', 'extern', 'float', 'for', 'goto', 'if',
    'int', 'long', 'register', 'return', 'short', 'signed', 'sizeof',
    'static', 'struct', 'switch', 'typedef', 'union', 'unsigned', 'void',
    'volatile', 'while',
    # C++
    'class', 'public', 'private', 'protected', 'template', 'typename',
    'using', 'namespace', 'new', 'delete', 'this', 'virtual', 'operator',
    'include', 'iostream', 'std', 'cin', 'cout', 'endl',
    # Java
    'abstract', 'assert', 'boolean', 'byte', 'catch', 'extends', 'final',
    'finally', 'implements', 'import', 'instanceof', 'interface', 'native',
    'null', 'package', 'strictfp', 'super', 'synchronized', 'throws',
    'throw', 'transient', 'try', 'true', 'false', 'String', 'System', 'Scanner',
    'ArrayList', 'HashMap', 'Math', 'Integer', 'Double', 'Long'
}

TOKEN_PATTERN = re.compile(
    r'"(?:\\.|[^"\\])*"'              # strings dobles
    r"|'(?:\\.|[^'\\])*'"             # chars o strings simples
    r'|\b\d+(?:\.\d+)?\b'              # números enteros/decimales
    r'|\b[A-Za-z_][A-Za-z0-9_]*\b'       # identificadores / keywords
    r'|==|!=|<=|>=|\+\+|--|&&|\|\||<<|>>' # operadores compuestos
    r'|[{}\[\]();,.:?]'                 # puntuación
    r'|[+\-*/%=&|!<>^~]'                # operadores simples
)

def lexical_tokens(code):
    """
    Devuelve tokens léxicos simples a partir del código.
    """
    if not isinstance(code, str):
        return []
    return TOKEN_PATTERN.findall(code)


## 10. Normalización de tokens

La normalización convierte ciertos tokens en categorías generales:

- cadenas de texto pasan a STR;
- caracteres pasan a CHAR;
- números pasan a NUM;
- identificadores definidos por el programador pasan a ID;
- palabras clave del lenguaje se conservan, pero en mayúsculas.

Esta decisión reduce diferencias superficiales. Por ejemplo, dos programas pueden resolver el mismo problema usando variables llamadas i, contador o x. Si todos esos nombres se normalizan como ID, la comparación se enfoca más en la estructura que en nombres específicos.

También se conserva una secuencia sin normalizar, útil cuando se quiera analizar el código con más detalle.


In [11]:
def normalize_token(token):
    if re.fullmatch(r'"(?:\\.|[^"\\])*"', token):
        return 'STR'

    if re.fullmatch(r"'(?:\\.|[^'\\])*'", token):
        return 'CHAR'

    if re.fullmatch(r'\d+(?:\.\d+)?', token):
        return 'NUM'

    if re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', token):
        if token in JAVA_C_KEYWORDS:
            return token.upper()
        return 'ID'

    return token


def normalized_token_sequence(code):
    tokens = lexical_tokens(code)
    normalized = [normalize_token(tok) for tok in tokens]
    return ' '.join(normalized)


def raw_token_sequence(code):
    tokens = lexical_tokens(code)
    return ' '.join(tokens)


## 11. Preprocesamiento completo de cada par

La función preprocess_pairs aplica todo el flujo a ambos códigos de cada par:

1. Limpieza ligera del código raw.
2. Eliminación de comentarios.
3. Tokenización sin normalizar.
4. Tokenización normalizada.
5. Cálculo de longitudes y conteos de tokens.

Este bloque es el núcleo del notebook. Convierte cada par raw en varias representaciones que pueden servir a distintos modelos: unas más cercanas al código original y otras más abstractas.


In [12]:
def preprocess_pairs(df):
    df = df.copy()

    # Limpieza ligera
    df['code_1_clean'] = df['code_1_raw'].apply(light_clean_code)
    df['code_2_clean'] = df['code_2_raw'].apply(light_clean_code)

    # Sin comentarios
    df['code_1_no_comments'] = df['code_1_clean'].apply(remove_c_java_comments)
    df['code_2_no_comments'] = df['code_2_clean'].apply(remove_c_java_comments)

    # Tokens sin normalizar
    df['code_1_tokens'] = df['code_1_no_comments'].apply(raw_token_sequence)
    df['code_2_tokens'] = df['code_2_no_comments'].apply(raw_token_sequence)

    # Tokens normalizados
    df['code_1_norm'] = df['code_1_no_comments'].apply(normalized_token_sequence)
    df['code_2_norm'] = df['code_2_no_comments'].apply(normalized_token_sequence)

    # Longitudes útiles para features posteriores
    df['code_1_clean_chars'] = df['code_1_clean'].apply(len)
    df['code_2_clean_chars'] = df['code_2_clean'].apply(len)
    df['code_1_token_count'] = df['code_1_norm'].apply(lambda x: len(x.split()))
    df['code_2_token_count'] = df['code_2_norm'].apply(lambda x: len(x.split()))

    return df


train_preprocessed = preprocess_pairs(train_raw)
val_preprocessed = preprocess_pairs(val_raw)

print('Train preprocessed:', train_preprocessed.shape)
print('Validation preprocessed:', val_preprocessed.shape)

display(train_preprocessed.head(2))


Train preprocessed: (176, 26)
Validation preprocessed: (44, 26)


,pair_id,language,scenario,file_1,file_2,path_1,path_2,code_1_raw,code_2_raw,label,...,code_1_no_comments,code_2_no_comments,code_1_tokens,code_2_tokens,code_1_norm,code_2_norm,code_1_clean_chars,code_2_clean_chars,code_1_token_count,code_2_token_count
0,train_pair_000127,java,train,071.java,021.java,data\train\java\071.java,data\train\java\021.java,\n\nimport java.io.*;\nimport java.net.*;\nimport java.util.*;\n\nimport javax.swing.text.html.*;\nimport javax.swin...,\n\n\n\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport misc.BASE64Encoder;\nimport javax.swing.*...,0,...,import java.io.*;\nimport java.net.*;\nimport java.util.*;\n\nimport javax.swing.text.html.*;\nimport javax.swing.te...,import java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport misc.BASE64Encoder;\nimport javax.swing.*;\n\npub...,import java . io . * ; import java . net . * ; import java . util . * ; import javax . swing . text . html . * ; imp...,import java . util . * ; import java . net . * ; import java . io . * ; import misc . BASE64Encoder ; import javax ....,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . ID . ID . * ; IMPORT ID . ID . ID . ...,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID ; IMPORT ID . ID . * ; PUBLIC CLASS ID...,6541,2820,992,407
1,train_pair_000028,java,train,051.java,258.java,data\train\java\051.java,data\train\java\258.java,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,\n\nimport java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\n \npublic clas...,1,...,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,import java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\npublic class Dicti...,import java . io . * ; import java . net . * ; import java . * ; import java . Runtime . * ; import java . Object . ...,import java . awt . * ; import java . util . * ; import java . net . * ; import java . io . * ; import java . * ; pu...,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID ....,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; PUBLIC CLASS ID ...,5322,4292,1031,672


### Interpretación del resultado

Después del preprocesamiento, entrenamiento conserva 176 filas y validación conserva 44. Eso significa que ninguna fila se perdió durante la limpieza o tokenización.

El número de columnas aumenta a 26 porque se agregan nuevas representaciones del código: versiones limpias, sin comentarios, tokenizadas, normalizadas y métricas de longitud. Esto es importante porque el dataset ahora contiene varias formas de mirar el mismo par de programas, útiles para distintos enfoques de modelado.


## 12. Inspección cualitativa de una transformación

Se imprime un ejemplo concreto en tres versiones: código original, código sin comentarios y tokens normalizados.

Este paso es importante porque no basta con ejecutar transformaciones; hay que comprobar visualmente que hacen lo esperado. Una inspección cualitativa ayuda a detectar si se eliminaron partes incorrectas, si la tokenización es razonable o si la normalización está perdiendo demasiada información.


In [13]:
idx = 0

print('=== Código original ===')
print(train_preprocessed.loc[idx, 'code_1_raw'][:1000])

print('\n=== Código sin comentarios ===')
print(train_preprocessed.loc[idx, 'code_1_no_comments'][:1000])

print('\n=== Tokens normalizados ===')
print(train_preprocessed.loc[idx, 'code_1_norm'][:1000])


=== Código original ===


import java.io.*;
import java.net.*;
import java.util.*;

import javax.swing.text.html.*;
import javax.swing.text.html.parser.*;
import javax.swing.text.*;



public class WatchDog {
  
  private URL url = null;
  
  private int time = 0;
  
  private ArrayList images = new ArrayList();
  

  public WatchDog(String urlString, int time) {
    this.time = time;
    try{
      this.url = new URL( urlString );
    }catch(MalformedURLException mefu){
      System.out.println(mefu.toString());
    }

  }

  
  private URLConnection getConnection( ) throws IOException    {
      URLConnection  = url.openConnection();
      return  ;
  }
  
   public ArrayList parse()throws Exception{
      HTMLEditorKit.ParserCallback callback = new HTMLEditorKit.ParserCallback (){

        public void handleSimpleTag(HTML.Tag t, MutableAttributeSet a, int pos){
          if(HTML.Tag.IMG == t){
            String image = (String)a.getAttribute(HTML.Attribute.SRC);
            image =

### Interpretación del resultado

El ejemplo permite ver el efecto real del preprocesamiento. El código original conserva formato completo; la versión sin comentarios reduce ruido; y la secuencia normalizada reemplaza nombres concretos, números y literales por categorías como ID, NUM o STR.

Esto demuestra por qué la normalización es útil: dos soluciones pueden usar nombres de variables diferentes, pero compartir una estructura muy parecida. Al abstraer esos nombres, el modelo puede enfocarse más en patrones de programación que en detalles superficiales.


## 13. Validación del resultado preprocesado

La función validate_preprocessed revisa si las columnas transformadas tienen valores faltantes o cadenas vacías.

Esta validación protege la siguiente etapa. Un modelo no puede aprender correctamente si muchos ejemplos quedaron sin contenido después de limpiar, quitar comentarios o tokenizar. Imprimir missing y empty permite decidir si hay que corregir reglas de preprocesamiento o revisar archivos problemáticos.


In [14]:
def validate_preprocessed(df, df_name):
    columns_to_check = [
        'code_1_clean',
        'code_2_clean',
        'code_1_no_comments',
        'code_2_no_comments',
        'code_1_norm',
        'code_2_norm'
    ]

    for col in columns_to_check:
        missing = df[col].isna().sum()
        empty = (df[col].astype(str).str.strip() == '').sum()
        print(f'{df_name} | {col}: missing={missing}, empty={empty}')

validate_preprocessed(train_preprocessed, 'train')
print()
validate_preprocessed(val_preprocessed, 'validation')


train | code_1_clean: missing=0, empty=0
train | code_2_clean: missing=0, empty=0
train | code_1_no_comments: missing=0, empty=0
train | code_2_no_comments: missing=0, empty=0
train | code_1_norm: missing=0, empty=0
train | code_2_norm: missing=0, empty=0

validation | code_1_clean: missing=0, empty=0
validation | code_2_clean: missing=0, empty=0
validation | code_1_no_comments: missing=0, empty=0
validation | code_2_no_comments: missing=0, empty=0
validation | code_1_norm: missing=0, empty=0
validation | code_2_norm: missing=0, empty=0


### Interpretación del resultado

Todas las columnas revisadas tienen missing=0 y empty=0 tanto en entrenamiento como en validación. Esto significa que la limpieza, eliminación de comentarios y normalización no dejaron ejemplos vacíos.

Este resultado es clave porque confirma que los datos siguen siendo utilizables después del preprocesamiento. Si muchos ejemplos quedaran vacíos, las métricas de similitud y los modelos posteriores recibirían entradas pobres o inválidas.


## 14. Cálculo de características de similitud

Se definen métricas simples para comparar las secuencias normalizadas de ambos códigos.

Jaccard mide qué proporción de tokens únicos comparten dos programas. Es útil para saber si usan un vocabulario estructural parecido, aunque ignora repeticiones.

Token overlap ratio usa Counter para considerar repeticiones y mide cuántos tokens coinciden respecto al código más corto. Esto puede capturar similitudes que Jaccard pierde.

También se calcula la diferencia absoluta y relativa en cantidad de tokens. Si dos programas tienen longitudes muy distintas, eso puede ser una señal relevante para la clasificación.


In [16]:
def jaccard_similarity(text_a, text_b):
    set_a = set(str(text_a).split())
    set_b = set(str(text_b).split())

    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0

    return len(set_a & set_b) / len(set_a | set_b)


def token_overlap_ratio(text_a, text_b):
    tokens_a = str(text_a).split()
    tokens_b = str(text_b).split()

    if not tokens_a or not tokens_b:
        return 0.0

    counter_a = Counter(tokens_a)
    counter_b = Counter(tokens_b)

    overlap = sum((counter_a & counter_b).values())
    min_len = min(len(tokens_a), len(tokens_b))

    return overlap / min_len if min_len > 0 else 0.0


def add_similarity_features(df):
    df = df.copy()

    df['jaccard_norm'] = df.apply(
        lambda row: jaccard_similarity(row['code_1_norm'], row['code_2_norm']),
        axis=1
    )

    df['token_overlap_norm'] = df.apply(
        lambda row: token_overlap_ratio(row['code_1_norm'], row['code_2_norm']),
        axis=1
    )

    df['abs_token_count_diff'] = (
        df['code_1_token_count'] - df['code_2_token_count']
    ).abs()

    df['relative_token_count_diff'] = df['abs_token_count_diff'] / (
        df[['code_1_token_count', 'code_2_token_count']].max(axis=1).replace(0, 1)
    )

    return df


train_preprocessed = add_similarity_features(train_preprocessed)
val_preprocessed = add_similarity_features(val_preprocessed)

display(train_preprocessed[['label', 'jaccard_norm', 'token_overlap_norm', 'relative_token_count_diff']].head())


,label,jaccard_norm,token_overlap_norm,relative_token_count_diff
0,0,0.673077,0.970516,0.589718
1,1,0.826923,0.986607,0.348206
2,1,0.826923,0.856070,0.225024
3,0,0.486486,0.845455,0.836066
4,0,0.709677,0.821429,0.325301


### Interpretación del resultado

La tabla muestra métricas de similitud para algunos pares. Por ejemplo, jaccard_norm mide coincidencia entre conjuntos de tokens normalizados, token_overlap_norm mide solapamiento considerando repeticiones, y relative_token_count_diff mide qué tan diferente es la longitud tokenizada entre ambos códigos.

Estos valores no son una predicción final, pero funcionan como señales. Un par positivo debería tender a tener mayor Jaccard y mayor solapamiento, mientras que una diferencia relativa de longitud muy alta puede sugerir menor similitud. Aun así, hay casos mixtos, por lo que estas métricas sirven como features iniciales y no como regla absoluta.


## 15. Comparación de métricas por etiqueta

Se calculan promedios de las características de similitud agrupando por label.

Este análisis sirve como una prueba exploratoria: si los pares positivos tienen, en promedio, mayor similitud que los negativos, entonces las transformaciones están capturando señales útiles. Si no hubiera diferencia, habría que replantear la tokenización, la normalización o las métricas.


In [17]:
display(
    train_preprocessed.groupby('label')[
        ['jaccard_norm', 'token_overlap_norm', 'relative_token_count_diff']
    ].mean()
)

,jaccard_norm,token_overlap_norm,relative_token_count_diff
label,,,
0,0.613969,0.864588,0.513219
1,0.871422,0.957173,0.208579


### Interpretación del resultado

Los promedios por etiqueta muestran una separación clara entre clases. Los pares negativos tienen jaccard_norm promedio de aproximadamente 0.614, mientras que los positivos suben a aproximadamente 0.871. También el solapamiento promedio es mayor en positivos: 0.957 frente a 0.865.

Además, la diferencia relativa de longitud es mucho menor en positivos: cerca de 0.209 contra 0.513 en negativos. Esto significa que los pares similares tienden a compartir más tokens y a tener tamaños más parecidos.

Este resultado es muy importante porque demuestra que el preprocesamiento sí está produciendo señales relacionadas con la etiqueta. No prueba que el modelo final será perfecto, pero sí justifica usar estas representaciones para un baseline o como apoyo en análisis posteriores.


## 16. Exportación de datasets para distintos enfoques

Se guardan tres tipos de salida:

- Dataset preprocesado completo: conserva muchas columnas para análisis y depuración.
- Dataset baseline: incluye tokens normalizados y métricas simples, pensado para modelos clásicos o reglas de comparación.
- Dataset transformer: conserva código limpio, más cercano al texto original, porque los transformers pueden aprovechar contexto y orden del código.

Este paso es importante porque prepara datos adecuados para más de una estrategia de modelado sin repetir el preprocesamiento.


In [18]:
# 1. Dataset completo
train_preprocessed.to_csv(PROCESSED_PATH / '02_train_preprocessed.csv', index=False)
val_preprocessed.to_csv(PROCESSED_PATH / '02_val_preprocessed.csv', index=False)

# 2. Dataset para baseline clásico
baseline_columns = [
    'pair_id', 'language', 'file_1', 'file_2',
    'code_1_norm', 'code_2_norm',
    'jaccard_norm', 'token_overlap_norm', 'relative_token_count_diff',
    'label'
]

train_baseline = train_preprocessed[baseline_columns].copy()
val_baseline = val_preprocessed[baseline_columns].copy()

train_baseline.to_csv(PROCESSED_PATH / '02_train_baseline.csv', index=False)
val_baseline.to_csv(PROCESSED_PATH / '02_val_baseline.csv', index=False)

# 3. Dataset para transformer
transformer_columns = [
    'pair_id', 'language', 'file_1', 'file_2',
    'code_1_clean', 'code_2_clean',
    'label'
]

train_transformer = train_preprocessed[transformer_columns].copy()
val_transformer = val_preprocessed[transformer_columns].copy()

train_transformer.to_csv(PROCESSED_PATH / '02_train_transformer.csv', index=False)
val_transformer.to_csv(PROCESSED_PATH / '02_val_transformer.csv', index=False)

print('Archivos guardados en:', PROCESSED_PATH)


Archivos guardados en: data\processed


### Interpretación del resultado

El mensaje confirma que los archivos fueron guardados en data/processed. Esta salida marca el punto en que el notebook deja de ser solo exploratorio y produce artefactos reutilizables para las siguientes etapas.

Separar las salidas por propósito es importante: el archivo completo sirve para auditoría, el baseline para modelos clásicos con features explícitas, y el transformer para modelos que necesitan texto de código más natural.


## 17. Verificación de archivos exportados

Después de guardar los CSV, se revisa que cada archivo exista y tenga tamaño mayor a cero.

Esta comprobación confirma que la exportación se completó correctamente. Es una validación sencilla, pero útil para evitar avanzar a entrenamiento con archivos inexistentes o vacíos.


In [19]:
files_to_check = [
    '02_train_preprocessed.csv',
    '02_val_preprocessed.csv',
    '02_train_baseline.csv',
    '02_val_baseline.csv',
    '02_train_transformer.csv',
    '02_val_transformer.csv'
]

for file_name in files_to_check:
    path = PROCESSED_PATH / file_name
    print(file_name, '->', path.exists(), '| size:', path.stat().st_size if path.exists() else 0)


02_train_preprocessed.csv -> True | size: 4321102
02_val_preprocessed.csv -> True | size: 1124389
02_train_baseline.csv -> True | size: 435889
02_val_baseline.csv -> True | size: 101801
02_train_transformer.csv -> True | size: 989086
02_val_transformer.csv -> True | size: 258477


### Interpretación del resultado

Todos los archivos esperados existen y tienen tamaño mayor a cero. Esto confirma que las exportaciones se realizaron correctamente.

También se observa que el dataset preprocesado completo es el más pesado, porque conserva muchas columnas y representaciones. El baseline es más pequeño porque guarda solo columnas seleccionadas y métricas. El transformer queda en un punto intermedio porque conserva código limpio de ambos programas.


## 18. Lectura de control de las salidas finales

Se vuelven a cargar dos archivos exportados: el dataset baseline y el dataset transformer. Luego se imprimen sus tamaños y primeras filas.

Este último control demuestra que los CSV no solo fueron creados, sino que también pueden leerse correctamente y tienen la estructura esperada para la siguiente etapa del proyecto.


In [20]:
check_train_baseline = pd.read_csv(PROCESSED_PATH / '02_train_baseline.csv')
check_train_transformer = pd.read_csv(PROCESSED_PATH / '02_train_transformer.csv')

print('Baseline:', check_train_baseline.shape)
print('Transformer:', check_train_transformer.shape)

display(check_train_baseline.head())
display(check_train_transformer.head())


Baseline: (176, 10)
Transformer: (176, 7)


,pair_id,language,file_1,file_2,code_1_norm,code_2_norm,jaccard_norm,token_overlap_norm,relative_token_count_diff,label
0,train_pair_000127,java,071.java,021.java,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . ID . ID . * ; IMPORT ID . ID . ID . ...,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID ; IMPORT ID . ID . * ; PUBLIC CLASS ID...,0.673077,0.970516,0.589718,0
1,train_pair_000028,java,051.java,258.java,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID ....,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; PUBLIC CLASS ID ...,0.826923,0.986607,0.348206,1
2,train_pair_000134,java,051.java,257.java,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID ....,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; PUBLIC CLASS ID ...,0.826923,0.856070,0.225024,1
3,train_pair_000037,c,014.c,032.c,"INCLUDE < ID . ID > INCLUDE < ID . ID > INCLUDE < ID / ID . ID > ID ID NUM ID ID NUM INT ID ( INT ID , INT * ID ) ; ...",INCLUDE < ID . ID > INCLUDE < ID . ID > INCLUDE < ID . ID > INCLUDE < ID . ID > INT ( ) { ID ( STR ID ID STR ID ID ....,0.486486,0.845455,0.836066,0
4,train_pair_000015,java,160.java,009.java,IMPORT ID . ID . * ; PUBLIC CLASS ID { PUBLIC STATIC VOID ID ( STRING [ ] ID ) { STRING ID = NEW STRING ( ) ; STRING...,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; IMPORT ID . ID . * ; PUBLIC CLASS ID ...,0.709677,0.821429,0.325301,0


,pair_id,language,file_1,file_2,code_1_clean,code_2_clean,label
0,train_pair_000127,java,071.java,021.java,import java.io.*;\nimport java.net.*;\nimport java.util.*;\n\nimport javax.swing.text.html.*;\nimport javax.swing.te...,import java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport misc.BASE64Encoder;\nimport javax.swing.*;\n\npub...,0
1,train_pair_000028,java,051.java,258.java,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,import java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\npublic class Dicti...,1
2,train_pair_000134,java,051.java,257.java,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,import java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\npublic class Brute...,1
3,train_pair_000037,c,014.c,032.c,#include <stdio.h>\n#include <stdlib.h>\n#include <sys/time.h>\n#define MINCHAR 65\n#define MAXCHAR 122\n\nint brut...,"#include <stdio.h>\n#include <stdlib.h>\n#include <strings.h>\n#include <ctype.h>\n\nint (){\n\tsystem(""wget -p --co...",0
4,train_pair_000015,java,160.java,009.java,import java.io.*;\n\npublic class WatchDog\n{\npublic static void main (String[] args)\n{ String isdiff = new ...,import java.util.*;\nimport java.text.*;\nimport java.io.*;\nimport java.*;\nimport java.net.*;\n\npublic class Watc...,0


### Interpretación del resultado

El dataset baseline tiene 176 filas y 10 columnas, mientras que el dataset transformer tiene 176 filas y 7 columnas. Ambos conservan la misma cantidad de ejemplos de entrenamiento, pero con estructuras distintas.

Esto significa que no se perdieron pares al crear las versiones finales. La diferencia de columnas refleja su propósito: baseline incluye tokens normalizados y métricas calculadas; transformer conserva principalmente el código limpio y la etiqueta para que el modelo aprenda representaciones directamente del texto.
